Import Libraries

In [6]:
! pip install pandas

  Attempting uninstall: numpy
    Found existing installation: numpy 1.18.1
    Uninstalling numpy-1.18.1:
      Successfully uninstalled numpy-1.18.1
  Attempting uninstall: pytz
    Found existing installation: pytz 2019.3
    Uninstalling pytz-2019.3:
      Successfully uninstalled pytz-2019.3
  Attempting uninstall: python-dateutil
    Found existing installation: python-dateutil 2.8.1
    Uninstalling python-dateutil-2.8.1:
      Successfully uninstalled python-dateutil-2.8.1


ERROR: chatterbot 1.0.5 has requirement python-dateutil<2.8,>=2.7, but you'll have python-dateutil 2.9.0.post0 which is incompatible.
ERROR: chatterbot 1.0.5 has requirement pyyaml<5.2,>=5.1, but you'll have pyyaml 3.13 which is incompatible.
ERROR: chatterbot 1.0.5 has requirement spacy<2.2,>=2.1, but you'll have spacy 2.2.3 which is incompatible.
ERROR: chatterbot 1.0.5 has requirement sqlalchemy<1.3,>=1.2, but you'll have sqlalchemy 1.3.12 which is incompatible.
You should consider upgrading via the 'c:\users\parmo\appdata\local\programs\python\python38\python.exe -m pip install --upgrade pip' command.


In [ ]:
import pandas as pd
import numpy as np
import os
import time

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [17]:
! pip install webdriver_manager


   ---------------------------------------- 0/2 [python-dotenv]
   -------------------- ------------------- 1/2 [webdriver_manager]
   -------------------- ------------------- 1/2 [webdriver_manager]
   -------------------- ------------------- 1/2 [webdriver_manager]
   -------------------- ------------------- 1/2 [webdriver_manager]
   ---------------------------------------- 2/2 [webdriver_manager]




[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### CONFIGURATION

In [2]:
BASE_URL = "https://www.screener.in/company"

CACHE_DIR = "data/fundamentals"

os.makedirs(CACHE_DIR, exist_ok=True)

In [9]:

BASE_PATH = "data/fundamentals"

os.makedirs(f"{BASE_PATH}/quarterly", exist_ok=True)
os.makedirs(f"{BASE_PATH}/annual", exist_ok=True)
os.makedirs(f"{BASE_PATH}/ratios", exist_ok=True)


### CREATE DRIVER

In [3]:
def create_driver():

    options = webdriver.ChromeOptions()

    options.add_argument("--headless")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--log-level=3")

    driver = webdriver.Chrome(options=options)

    return driver

### GENERIC TABLE PARSER

In [4]:
def parse_table(section):

    table = section.find_element(By.TAG_NAME, "table")

    rows = table.find_elements(By.TAG_NAME, "tr")

    headers = [
        th.text.strip()
        for th in rows[0].find_elements(By.TAG_NAME, "th")
    ]

    data = []

    for row in rows[1:]:

        cols = [
            td.text.strip()
            for td in row.find_elements(By.TAG_NAME, "td")
        ]

        if cols:
            data.append(cols)

    df = pd.DataFrame(data, columns=headers)

    return df

### CLEAN TABLE

In [5]:
def clean_financial_table(df):

    df = df.copy()

    # transpose
    df = df.set_index(df.columns[0]).T

    # remove commas
    df = df.replace(",", "", regex=True)

    # convert numeric
    for col in df.columns:

        df[col] = pd.to_numeric(
            df[col],
            errors="ignore"
        )

    # parse date index
    df.index = pd.to_datetime(
        df.index,
        errors="coerce"
    )

    df = df.sort_index()

    return df

In [15]:
def fetch_quarterly_profit_loss(company_code):

    df = pd.DataFrame()

    try:

        url = (
            f"https://www.screener.in/company/"
            f"{company_code}/consolidated/"
        )

        # your selenium logic here

        # finally create dataframe
        df = pd.DataFrame(data)

        return df

    except Exception as e:

        print(f"❌ Quarterly fetch failed: {e}")

        return df


In [11]:
def fetch_annual_profit_loss(code):

    url = f"https://www.screener.in/company/{code}/consolidated/#profit-loss"

    # selenium loading logic

    return df


In [12]:
def fetch_ratios(code):

    url = f"https://www.screener.in/company/{code}/"

    # extract:
    # PE
    # Market Cap
    # ROCE
    # ROE
    # Book Value
    # Face Value
    # Dividend Yield
    # etc

    return ratio_df


In [14]:
# ==========================================
# MAIN PIPELINE
# ==========================================

all_quarterly = []
all_annual = []
all_ratios = []

COMPANIES = {
    "HDBFS": {
        "screener": "HDB-Financial-Services",
        "name": "HDB Financial Services"
    },

    "PFC": {
        "screener": "PFC",
        "name": "Power Finance Corporation"
    },

    "TCS": {
        "screener": "TCS",
        "name": "Tata Consultancy Services"
    },

    "CRAMC": {
        "screener": "CRAMC",
        "name": "Canara Robeco Asset"
    },

    "SUZLON": {
        "screener": "SUZLON",
        "name": "Suzlon Energy"
    }
}


for ticker, info in COMPANIES.items():

    screener_code = info["screener"]
    company_name = info["name"]

    print(f"\n📊 Processing {ticker}")

    try:

        # ======================================
        # QUARTERLY
        # ======================================
        q_df = fetch_quarterly_profit_loss(
            screener_code
        )

        q_df["TICKER"] = ticker
        q_df["COMPANY"] = company_name

        q_path = (
            f"{BASE_PATH}/quarterly/"
            f"{ticker}_quarterly.csv"
        )

        q_df.to_csv(q_path, index=False)

        all_quarterly.append(q_df)

        print("✅ Quarterly Saved")


        # ======================================
        # ANNUAL
        # ======================================
        a_df = fetch_annual_profit_loss(
            screener_code
        )

        a_df["TICKER"] = ticker
        a_df["COMPANY"] = company_name

        a_path = (
            f"{BASE_PATH}/annual/"
            f"{ticker}_annual.csv"
        )

        a_df.to_csv(a_path, index=False)

        all_annual.append(a_df)

        print("✅ Annual Saved")


        # ======================================
        # RATIOS
        # ======================================
        r_df = fetch_ratios(
            screener_code
        )

        r_df["TICKER"] = ticker
        r_df["COMPANY"] = company_name

        r_path = (
            f"{BASE_PATH}/ratios/"
            f"{ticker}_ratios.csv"
        )

        r_df.to_csv(r_path, index=False)

        all_ratios.append(r_df)

        print("✅ Ratios Saved")

    except Exception as e:

        print(f"❌ Failed {ticker}: {e}")



📊 Processing HDBFS
❌ Failed HDBFS: name 'df' is not defined

📊 Processing PFC
❌ Failed PFC: name 'df' is not defined

📊 Processing TCS
❌ Failed TCS: name 'df' is not defined

📊 Processing CRAMC
❌ Failed CRAMC: name 'df' is not defined

📊 Processing SUZLON
❌ Failed SUZLON: name 'df' is not defined


In [ ]:
# ==========================================
# MERGE ALL
# ==========================================

if all_quarterly:

    pd.concat(
        all_quarterly,
        ignore_index=True
    ).to_csv(
        f"{BASE_PATH}/ALL_QUARTERLY.csv",
        index=False
    )

if all_annual:

    pd.concat(
        all_annual,
        ignore_index=True
    ).to_csv(
        f"{BASE_PATH}/ALL_ANNUAL.csv",
        index=False
    )

if all_ratios:

    pd.concat(
        all_ratios,
        ignore_index=True
    ).to_csv(
        f"{BASE_PATH}/ALL_RATIOS.csv",
        index=False
    )

print("\n✅ ALL FILES GENERATED")


### STANDARDIZE COLUMNS

In [6]:
def standardize_columns(df):

    df.columns = (
        df.columns
        .str.strip()
        .str.upper()
        .str.replace("%", "_PERCENT")
        .str.replace(" ", "_")
        .str.replace("+", "")
    )

    return df


### GENERIC SECTION FETCHER

In [7]:
def get_screener_section(
    driver,
    ticker,
    section_id
):

    url = f"{BASE_URL}/{ticker}/consolidated/"

    driver.get(url)

    wait = WebDriverWait(driver, 15)

    section = wait.until(
        EC.presence_of_element_located(
            (By.ID, section_id)
        )
    )

    df = parse_table(section)

    df = clean_financial_table(df)

    df = standardize_columns(df)

    return df

### FULL FUNDAMENTAL DOWNLOAD

In [8]:
def download_company_fundamentals(
    ticker,
    company_name=None
):

    print(f"\n📥 Downloading {ticker}")

    driver = create_driver()

    sections = {
        "quarterly": "quarters",
        "profit_loss": "profit-loss",
        "balance_sheet": "balance-sheet",
        "cash_flow": "cash-flow",
        "shareholding": "shareholding"
    }

    data = {}

    for name, section_id in sections.items():

        try:

            df = get_screener_section(
                driver,
                ticker,
                section_id
            )

            df["TICKER"] = ticker

            if company_name:
                df["COMPANY"] = company_name

            data[name] = df

            print(f"✅ {name}")

            time.sleep(1)

        except Exception as e:

            print(f"❌ {name}: {e}")

    driver.quit()

    return data